# 02 · Puntos de quiebre: qué cambia antes y después

**Qué hace.** Corta cada partido del América en **puntos de quiebre** (goles, expulsiones, cambios de sistema, sustituciones tácticas y por lesión, de ambos equipos) y mide qué cambia en los 10 minutos siguientes frente a los 10 anteriores, **en los dos equipos a la vez**. Descuenta lo que cambia solo por el paso del tiempo con **ventanas de control** (otros partidos del mismo entrenador, mismo periodo, mismo marcador, minuto ± 5, sin quiebre). Compara a Jardine con sus predecesores y prueba la **hipótesis de agresividad** (presión en un tramo → qué pasa en el siguiente, y si cambia después del 65').

**Qué necesita.** Pasos 02–05 corridos: `data/processed/` con la columna `tarjeta` (`python scripts/03_procesar.py --rehacer eventos`), `outputs/voronoi/metricas/` (04) y `outputs/panel/` (`python scripts/05_panel.py --procesos 4`).

**Cómo probarlo.** En VS Code, kernel `.venv`, *Run All*. Tarda 1–3 minutos. Las figuras y la tabla de resultados se guardan en `outputs/exploracion/` y `outputs/hallazgos/` (local, no se suben).

**Criterio de hallazgo, versión 2** (18 oct, tras la auditoría A2; `contexto/framework.md`). Un efecto es hallazgo si cumple las cuatro condiciones:
1. Su IC 95 % por bootstrap de partidos no cruza 0.
2. Mide al menos 0.5 DE del **cambio en las ventanas de control**. En la v1 del 12 oct era la DE entre partidos, que en ventanas de 10 minutos casi no filtraba.
3. Va en la misma dirección en al menos 4 torneos de Jardine.
4. **Sobrevive a Benjamini-Hochberg** (q ≤ `hallazgo.fdr_q`) dentro de su familia de pruebas.

Una **prueba placebo** aplica el mismo criterio a ventanas de control tratadas como si fueran quiebres y dice cuántas pruebas pasan por azar. La columna `pasa_v1` conserva el resultado de la versión 1 para comparar.

> IMPOR1: el **Clausura 2026 es el torneo de prueba** y aquí no se usa (`INCLUIR_PRUEBA = False`): los hallazgos alimentan los principios que luego se validan ahí. Con 5 torneos de entrenamiento (AP23–AP25), la condición 3 pide 4 de 5.
>
> IMPOR2: solo agregados. El notebook se sube sin salidas (nbstripout).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from ojo.config import DIR_SALIDAS, params
from ojo.datos import almacen
from ojo.framework import contexto, quiebres
from ojo.framework import partidos as fp
from ojo.lentes import hallazgo, minuto
from ojo.salidas.estilo import color

P = params()
EQUIPO = P["equipo"]["team_id"]
HALLAZGO, AGRESIVIDAD = P["hallazgo"], P["agresividad"]
RNG = np.random.default_rng(P["simulacion"]["semilla"])
INCLUIR_PRUEBA = False  # IMPOR1: el CL26 solo se abre al final, para confirmar
DIR_FIG = DIR_SALIDAS / "exploracion"
DIR_HALLAZGOS = DIR_SALIDAS / "hallazgos"
for d in (DIR_FIG, DIR_HALLAZGOS):
    d.mkdir(parents=True, exist_ok=True)

ENTRENADORES = {d["manager_id"]: d["nombre"].split()[-1] for d in P["predecesores"]}
ENTRENADORES[P["entrenador"]["manager_id"]] = P["entrenador"]["nombre"].split()[-1]
NUEVO, PREVIOS = list(ENTRENADORES.values())[-1], "Predecesores"
METRICAS = [m for lista in minuto.METRICAS.values() for m in lista]
NOMBRES = {
    "control_campo_rival": "Control en campo rival", "altura_sin_balon": "Altura del bloque sin balón",
    "largo_sin_balon": "Largo del bloque sin balón", "ancho_con_balon": "Ancho con balón",
    "compactacion_sin_balon": "Compactación sin balón (yd²/jugador)", "centralizacion": "Centralización de la red",
    "presiones": "Presiones/min", "faltas": "Faltas/min", "contrapresiones": "Contrapresiones/min",
    "xg": "xG/min", "obv": "OBV/min", "field_tilt": "Field tilt",
    "tiros_transicion_contra": "Tiros en transición en contra/min", "contragolpes_contra": "Contragolpes en contra/min",
}
METRICAS

## 1. Partidos

Partidos válidos del América con Solari, Ortiz o Jardine (sin interinos), sin el torneo de prueba. *Predecesores* = Solari + Ortiz.

In [ ]:
partidos = fp.con_torneo(fp.del_equipo(fp.validos(almacen.cargar("partidos")), EQUIPO))
partidos = partidos[partidos["entrenador_id"].isin(ENTRENADORES)].copy()
partidos["prueba"] = fp.es_prueba(partidos)
if not INCLUIR_PRUEBA:
    partidos = partidos[~partidos["prueba"]]
partidos["entrenador"] = partidos["entrenador_id"].map(ENTRENADORES)
partidos["grupo"] = np.where(partidos["entrenador"] == NUEVO, NUEVO, PREVIOS)
TORNEOS = list(partidos.sort_values("fecha")["torneo_corto"].unique())
TORNEOS_NUEVO = [t for t in TORNEOS if t in set(partidos.loc[partidos["grupo"] == NUEVO, "torneo_corto"])]
ids = partidos["match_id"].astype(int).tolist()

panel = minuto.cargar_panel("minuto").query("match_id in @ids")
aristas = minuto.cargar_panel("aristas").query("match_id in @ids")
print(f"Torneos de {NUEVO} usados: {', '.join(TORNEOS_NUEVO)}")
partidos.groupby("entrenador", sort=False).agg(partidos=("match_id", "size"),
                                               torneos=("torneo_corto", lambda s: ", ".join(s.unique())))

## 2. Quiebres y ventanas

Cada quiebre tiene una ventana antes y una después (10 min, sin el minuto del quiebre ni el de al lado). Las ventanas se cortan en el quiebre que cambia el marcador o el número de jugadores más cercano; si una queda en menos de 4 minutos, el quiebre no se usa (`valida`).

In [ ]:
eventos = almacen.cargar("eventos", ids, columnas=["match_id", "indice", "periodo", "minuto", "tipo", "equipo_id",
                                                  "tiro_resultado", "tarjeta", "formacion", "sustitucion_resultado"])
limites = quiebres.limites_periodo(eventos)
q = quiebres.detectar(eventos, EQUIPO).merge(partidos[["match_id", "grupo", "entrenador", "torneo_corto"]], on="match_id")
q = quiebres.ventanas(q, limites)
conteo = q.groupby(["tipo", "grupo"]).agg(quiebres=("valida", "size"), validos=("valida", "sum")).unstack("grupo")
conteo.reindex(quiebres.TIPOS)

## 3. Ventanas de control

Para cada quiebre válido: hasta 20 ventanas de **otros partidos del mismo grupo** (Jardine con Jardine, predecesores con predecesores), en el mismo periodo, con el mismo estado del marcador y minuto ± 5, sin quiebres que corten dentro. Cuánto cambian ahí las métricas es lo que habría cambiado sin el quiebre.

In [ ]:
segmentos = quiebres.segmentos_libres(q, limites).merge(partidos[["match_id", "grupo"]], on="match_id")
controles = quiebres.controles(q, segmentos, por=("grupo",))
controles["id"] = controles["quiebre_id"] + "|c" + controles.groupby("quiebre_id").cumcount().astype(str)
por_quiebre = controles.groupby("quiebre_id").size()
validos = q[q["valida"]]
print(f"Quiebres válidos: {len(validos):,}; con al menos un control: {validos['quiebre_id'].isin(por_quiebre.index).sum():,}; "
      f"controles por quiebre: mediana {por_quiebre.median():.0f}")

## 4. Métricas de cada ventana, a dos bandas

`propio` = América y `rival` = el rival en la misma ventana. Las tasas van por minuto; las métricas de estructura promedian los frames 360 de la ventana.

In [ ]:
def a_ventanas(tabla: pd.DataFrame) -> pd.DataFrame:
    # Ventanas antes y después de cada fila (id, match_id, periodo y límites) en formato largo.
    return pd.concat([pd.DataFrame({"ventana_id": tabla["id"] + "|" + fase, "match_id": tabla["match_id"],
                                    "periodo": tabla["periodo"], "desde": tabla[f"{fase}_desde"],
                                    "hasta": tabla[f"{fase}_hasta"]})
                      for fase in ("antes", "despues")], ignore_index=True)


ventanas = pd.concat([a_ventanas(validos.assign(id=validos["quiebre_id"])), a_ventanas(controles)], ignore_index=True)
medidas = minuto.medir(panel, aristas, ventanas)
medidas[["id", "fase"]] = medidas["ventana_id"].str.rsplit("|", n=1, expand=True)
largo = medidas.melt(id_vars=["id", "fase", "lado"], value_vars=METRICAS, var_name="metrica")
cambio = largo.pivot_table(index=["id", "lado", "metrica"], columns="fase", values="value").reset_index()
cambio["cambio"] = cambio["despues"] - cambio["antes"]
cambio["quiebre_id"] = cambio["id"].str.split("|c", regex=False).str[0]
cambio["es_control"] = cambio["id"].str.contains("|c", regex=False)

base_control = (cambio[cambio["es_control"]].groupby(["quiebre_id", "lado", "metrica"])["cambio"]
                .agg(cambio_control="mean", n_controles="size"))
efectos = (cambio[~cambio["es_control"]].join(base_control, on=["quiebre_id", "lado", "metrica"])
           .merge(q[["quiebre_id", "match_id", "tipo", "grupo", "torneo_corto", "minuto", "estado_antes"]], on="quiebre_id"))
efectos["efecto"] = efectos["cambio"] - efectos["cambio_control"]
print(f"Ventanas medidas: {len(medidas) // 2:,}; efectos (quiebre × lado × métrica): {efectos['efecto'].notna().sum():,}")
efectos.groupby(["grupo", "lado"])["efecto"].count().unstack()

## 5. Variación entre partidos

La condición 2 compara cada efecto con una desviación estándar.

- **v1:** la de la métrica entre partidos, con el valor del partido completo.
- **v2:** la del cambio antes→después en las ventanas de control. Es la escala correcta para un efecto de ventanas de 10 minutos (auditoría A2).

In [ ]:
minutos_partido = panel[minuto.CLAVES_MINUTO].drop_duplicates().assign(partido=lambda d: d["match_id"])
por_partido = minuto.metricas(minuto.sumar(panel, minutos_partido, "partido"), "partido",
                              minuto.ventanas_red(aristas, minutos_partido, "partido"))
DE = por_partido.groupby("lado")[METRICAS].std().T  # v1: entre partidos
DE_CONTROL = cambio[cambio["es_control"]].groupby(["metrica", "lado"])["cambio"].std().unstack("lado")  # v2
pd.concat({"v1: entre partidos": DE, "v2: cambio en las ventanas de control": DE_CONTROL}, axis=1).round(4)

## 6. Efecto de cada quiebre con Jardine

Una fila por tipo de quiebre, lado y métrica: efecto medio (cambio tras el quiebre menos el de sus controles), IC 95 % por bootstrap de partidos, tamaño en DE entre partidos y en cuántos torneos va en la misma dirección. `pasa` = cumple las cuatro condiciones del criterio v2; `pasa_v1` = el criterio del 12 oct; `q` = valor de Benjamini-Hochberg; `muestra_chica` = menos de 10 partidos (aviso, no condición).

In [ ]:
def evaluar_grupo(tabla: pd.DataFrame) -> pd.DataFrame:
    # Criterio v2 a cada (tipo, lado, métrica): condiciones 1–3 con la escala de los controles, p del bootstrap y FDR.
    filas = []
    for (tipo, lado, metrica), g in tabla.dropna(subset=["efecto"]).groupby(["tipo", "lado", "metrica"]):
        medias = hallazgo.medias_bootstrap(g, "efecto", RNG)
        bajo, alto = hallazgo.intervalo(medias)
        efecto = g.groupby("match_id")["efecto"].sum().sum() / len(g)
        misma = hallazgo.direcciones(g.groupby("torneo_corto")["efecto"].mean(), efecto)
        filas.append({"tipo": tipo, "lado": lado, "metrica": metrica, "quiebres": len(g),
                      "partidos": g["match_id"].nunique(), "efecto": efecto, "ic_bajo": bajo, "ic_alto": alto,
                      "torneos_misma_direccion": misma,
                      "muestra_chica": g["match_id"].nunique() < HALLAZGO["partidos_aviso"],
                      "p": hallazgo.p_bootstrap(medias),
                      "pasa_v1": hallazgo.evaluar(efecto, bajo, alto, DE.loc[metrica, lado], misma)["pasa"],
                      **hallazgo.evaluar(efecto, bajo, alto, DE_CONTROL.loc[metrica, lado], misma)})
    return hallazgo.con_fdr(pd.DataFrame(filas))


jardine = evaluar_grupo(efectos[efectos["grupo"] == NUEVO])
jardine.to_parquet(DIR_HALLAZGOS / "quiebres_jardine.parquet", index=False)
print(f"Pruebas: {len(jardine)} · IC sin cero: {jardine['ic_sin_cero'].sum()} · v1: {jardine['pasa_v1'].sum()} · "
      f"condiciones 1–3 con la escala v2: {jardine['pasa_1a3'].sum()} · v2 (con FDR): {jardine['pasa'].sum()}")
columnas = ["tipo", "lado", "metrica", "quiebres", "partidos", "efecto", "ic_bajo", "ic_alto", "tamano_de",
            "torneos_misma_direccion", "q", "muestra_chica", "pasa_v1", "pasa"]
jardine.sort_values(["pasa", "ic_sin_cero", "tamano_de"], ascending=[False, False, False],
                    key=lambda s: s.abs() if s.name == "tamano_de" else s)[columnas].head(20).round(3)

### Mapa de efectos

Tamaño del efecto en DE entre partidos (dorado = sube tras el quiebre, azul = baja). Con borde blanco: pasa el criterio; con punto: el IC no cruza 0 pero falla otra condición.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

mapa = LinearSegmentedColormap.from_list("oda", [color("azul"), color("gris-fondo"), color("dorado")])
fig, ejes = plt.subplots(1, 2, figsize=(15, 6.5), sharey=True)
for eje, lado in zip(ejes, ("propio", "rival")):
    sub = jardine[jardine["lado"] == lado]
    tam = sub.pivot(index="tipo", columns="metrica", values="tamano_de").reindex(index=quiebres.TIPOS, columns=METRICAS)
    imagen = eje.imshow(tam.to_numpy(dtype=float), cmap=mapa, vmin=-1, vmax=1, aspect="auto")
    for (i, tipo) in enumerate(tam.index):
        for (j, metrica) in enumerate(tam.columns):
            fila = sub[(sub["tipo"] == tipo) & (sub["metrica"] == metrica)]
            if fila.empty:
                continue
            if bool(fila["pasa"].iloc[0]):
                eje.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, ec="white", lw=2.5))
            elif bool(fila["ic_sin_cero"].iloc[0]):
                eje.plot(j, i, "o", color=color("gris-texto"), ms=3)
    eje.set_xticks(range(len(METRICAS)), [NOMBRES[m] for m in METRICAS], rotation=60, ha="right", fontsize=8)
    eje.set_yticks(range(len(quiebres.TIPOS)), quiebres.TIPOS, fontsize=8)
    eje.set_title(f"{'América' if lado == 'propio' else 'Rival'} · {NUEVO}", fontsize=11)
fig.colorbar(imagen, ax=ejes, shrink=.7, label="Efecto (DE entre partidos)")
fig.savefig(DIR_FIG / "02_mapa_efectos.png", dpi=130, bbox_inches="tight")
plt.show()

## 7. Jardine frente a sus predecesores

Diferencia del efecto de cada quiebre (Jardine − predecesores), con bootstrap de partidos independiente en cada grupo. La condición 3 compara el efecto de Jardine en cada uno de sus torneos con el de los predecesores.

In [ ]:
filas = []
for (tipo, lado, metrica), g in efectos.dropna(subset=["efecto"]).groupby(["tipo", "lado", "metrica"]):
    nuevo, previo = g[g["grupo"] == NUEVO], g[g["grupo"] == PREVIOS]
    if nuevo["match_id"].nunique() < 2 or previo["match_id"].nunique() < 2:
        continue
    medias = hallazgo.medias_bootstrap(nuevo, "efecto", RNG) - hallazgo.medias_bootstrap(previo, "efecto", RNG)
    bajo, alto = hallazgo.intervalo(medias)
    diferencia = nuevo["efecto"].mean() - previo["efecto"].mean()
    por_torneo = nuevo.groupby("torneo_corto")["efecto"].mean() - previo["efecto"].mean()
    misma = hallazgo.direcciones(por_torneo, diferencia)
    filas.append({"tipo": tipo, "lado": lado, "metrica": metrica, "quiebres_nuevo": len(nuevo),
                  "quiebres_previos": len(previo), "efecto_nuevo": nuevo["efecto"].mean(),
                  "efecto_previos": previo["efecto"].mean(), "diferencia": diferencia, "ic_bajo": bajo, "ic_alto": alto,
                  "torneos_misma_direccion": misma,
                  "muestra_chica": min(nuevo["match_id"].nunique(), previo["match_id"].nunique()) < HALLAZGO["partidos_aviso"],
                  "p": hallazgo.p_bootstrap(medias),
                  "pasa_v1": hallazgo.evaluar(diferencia, bajo, alto, DE.loc[metrica, lado], misma)["pasa"],
                  **hallazgo.evaluar(diferencia, bajo, alto, DE_CONTROL.loc[metrica, lado], misma)})
contra_previos = hallazgo.con_fdr(pd.DataFrame(filas))
contra_previos.to_parquet(DIR_HALLAZGOS / "quiebres_contra_predecesores.parquet", index=False)
print(f"Pruebas: {len(contra_previos)} · IC sin cero: {contra_previos['ic_sin_cero'].sum()} · v1: {contra_previos['pasa_v1'].sum()} · "
      f"condiciones 1–3 con la escala v2: {contra_previos['pasa_1a3'].sum()} · v2 (con FDR): {contra_previos['pasa'].sum()}")
contra_previos.sort_values(["pasa", "ic_sin_cero"], ascending=False).head(15).round(3)

### Prueba placebo (criterio v2)

Cada quiebre de Jardine con 2 o más ventanas de control presta una de ellas como **quiebre falso**; las demás hacen de controles. Ahí no pasó nada, así que lo que pase el criterio es azar: es la tasa de falsos positivos del método.

In [ ]:
controles_info = controles[["id", "match_id"]].merge(partidos[["match_id", "torneo_corto"]], on="match_id")
falsos = (hallazgo.efectos_placebo(cambio[cambio["es_control"]])
          .merge(controles_info, on="id")
          .merge(q[["quiebre_id", "tipo", "grupo"]], on="quiebre_id"))
placebo = evaluar_grupo(falsos[falsos["grupo"] == NUEVO])
placebo.to_parquet(DIR_HALLAZGOS / "placebo.parquet", index=False)
print(f"Placebo con {NUEVO}: {len(placebo)} pruebas · IC sin cero: {placebo['ic_sin_cero'].sum()} · v1: {placebo['pasa_v1'].sum()} · "
      f"1–3 con la escala v2: {placebo['pasa_1a3'].sum()} · v2 (con FDR): {placebo['pasa'].sum()}")

## 8. Cómo se ve un quiebre minuto a minuto

Promedio por minuto alrededor de los goles (minuto 0 = el gol), América y rival, frente a sus ventanas de control. Es la forma del efecto; la prueba formal es la de las secciones 6 y 7.

In [ ]:
RELATIVOS = range(-P["quiebres"]["ventana_min"], P["quiebres"]["ventana_min"] + 1)


def trayectoria(tabla: pd.DataFrame, columna: str) -> pd.DataFrame:
    # Promedio de una columna del panel por minuto relativo y lado (tabla: match_id, periodo, minuto de referencia).
    filas = tabla[["match_id", "periodo", "minuto"]].rename(columns={"minuto": "m0"})
    filas = filas.merge(pd.DataFrame({"relativo": list(RELATIVOS)}), how="cross")
    filas["minuto"] = filas["m0"] + filas["relativo"]
    unidos = filas.merge(panel[[*minuto.CLAVES_MINUTO, "lado", columna]], on=minuto.CLAVES_MINUTO)
    return unidos.groupby(["relativo", "lado"])[columna].mean().unstack("lado")


fig, ejes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for eje, tipo in zip(ejes, ("gol_favor", "gol_contra")):
    sel = validos[(validos["tipo"] == tipo) & (validos["grupo"] == NUEVO)]
    ctrl = controles[controles["quiebre_id"].isin(sel["quiebre_id"])]
    for tabla, estilo, etiqueta in ((sel, "-", "tras el gol"), (ctrl, ":", "control")):
        t = trayectoria(tabla, "presiones").rolling(3, center=True, min_periods=1).mean()
        eje.plot(t.index, t["propio"], estilo, color=color("dorado"), lw=2, label=f"América · {etiqueta}")
        eje.plot(t.index, t["rival"], estilo, color=color("azul"), lw=2, label=f"Rival · {etiqueta}")
    eje.axvspan(-1.5, 1.5, color="0.9")
    eje.set_title(f"{tipo.replace('_', ' ')} ({len(sel)} con {NUEVO})")
    eje.set_xlabel("Minutos desde el gol")
    eje.set_xticks(range(min(RELATIVOS), max(RELATIVOS) + 1, 2))
ejes[0].set_ylabel("Presiones por minuto (media móvil de 3)")
ejes[1].legend(fontsize=8, frameon=False)
fig.savefig(DIR_FIG / "02_presion_alrededor_de_goles.png", dpi=130, bbox_inches="tight")
plt.show()

## 9. Hipótesis de agresividad

Tramos de 5 minutos dentro de cada periodo. ¿La presión del América en campo rival en un tramo se relaciona con lo que pasa en el **siguiente**: su xG, sus faltas, y los contragolpes y tiros en transición del rival? Modelo con efectos fijos por partido y por marcador, la respuesta del tramo actual como control, errores agrupados por partido. `tardío` = tramos desde el 65'. El coeficiente está en unidades de la respuesta por **+1 DE de presión**. Es una relación temporal, no causalidad.

In [ ]:
T, TARDIO = AGRESIVIDAD["tramo_min"], AGRESIVIDAD["tardio_min"]
pm = panel.copy()
inicio, fin = pm["periodo"].map(contexto.INICIO_PERIODO), pm["periodo"].map(contexto.FIN_PERIODO)
pm["k"] = (np.minimum(pm["minuto"], fin) - inicio) // T  # el añadido queda como un tramo propio
claves = ["match_id", "periodo", "k"]
sumas = pm.groupby([*claves, "lado"])[["presiones_campo_rival", "xg", "faltas", "contragolpes", "tiros_transicion"]].sum()
sumas = sumas.unstack("lado", fill_value=0)
n_min = pm.groupby(claves)["minuto"].nunique()
tramos = pd.DataFrame({
    "presion": sumas[("presiones_campo_rival", "propio")] / n_min,
    "xg_favor": sumas[("xg", "propio")] / n_min,
    "faltas": sumas[("faltas", "propio")] / n_min,
    "contragolpes_contra": sumas[("contragolpes", "rival")] / n_min,
    "transicion_contra": sumas[("tiros_transicion", "rival")] / n_min,
    "estado": pm.groupby(claves)["estado"].first(),
    "inicio_tramo": (inicio + pm["k"] * T).groupby([pm[c] for c in claves]).first(),
}).reset_index().merge(partidos[["match_id", "grupo", "torneo_corto"]], on="match_id")
RESPUESTAS = ["xg_favor", "faltas", "contragolpes_contra", "transicion_contra"]
siguiente = tramos.groupby(["match_id", "periodo"])[RESPUESTAS].shift(-1).add_suffix("_sig")
tramos = pd.concat([tramos, siguiente], axis=1).dropna(subset=[f"{r}_sig" for r in RESPUESTAS])
tramos["tardio"] = (tramos["inicio_tramo"] >= TARDIO).astype(int)
tramos["presion_z"] = (tramos["presion"] - tramos["presion"].mean()) / tramos["presion"].std()
DE_PARTIDO = {"xg_favor": DE.loc["xg", "propio"], "faltas": DE.loc["faltas", "propio"],
              "contragolpes_contra": DE.loc["contragolpes_contra", "propio"],
              "transicion_contra": DE.loc["tiros_transicion_contra", "propio"]}


def ajustar(datos: pd.DataFrame, respuesta: str):
    formula = f"{respuesta}_sig ~ presion_z * tardio + {respuesta} + C(estado) + C(match_id)"
    return smf.ols(formula, datos).fit(cov_type="cluster", cov_kwds={"groups": datos["match_id"]})


filas = []
for grupo in (NUEVO, PREVIOS):
    datos = tramos[tramos["grupo"] == grupo]
    for r in RESPUESTAS:
        modelo = ajustar(datos, r)
        for momento, combinacion in (("antes del 65'", "presion_z = 0"), ("desde el 65'", "presion_z + presion_z:tardio = 0")):
            prueba = modelo.t_test(combinacion)
            coef, (bajo, alto) = float(prueba.effect[0]), prueba.conf_int(alpha=1 - P["estadistica"]["nivel_confianza"])[0]
            misma = 0
            if grupo == NUEVO:
                por_torneo = pd.Series({t: float(ajustar(d, r).t_test(combinacion).effect[0])
                                        for t, d in datos.groupby("torneo_corto")})
                misma = hallazgo.direcciones(por_torneo, coef)
            filas.append({"grupo": grupo, "respuesta": r, "momento": momento, "coef_por_DE_presion": coef,
                          "ic_bajo": bajo, "ic_alto": alto, "tramos": int(modelo.nobs), "torneos_misma_direccion": misma,
                          "p": float(np.asarray(prueba.pvalue).ravel()[0]),
                          **hallazgo.evaluar(coef, bajo, alto, DE_PARTIDO[r], misma)})
agresividad = pd.DataFrame(filas)
agresividad.loc[agresividad["grupo"] == PREVIOS, "pasa"] = False  # el criterio se evalúa en Jardine
de_jardine = agresividad["grupo"] == NUEVO
agresividad = pd.concat([hallazgo.con_fdr(agresividad[de_jardine]),
                         agresividad[~de_jardine].assign(pasa_1a3=False, q=np.nan, fdr=False)], ignore_index=True)
agresividad.to_parquet(DIR_HALLAZGOS / "agresividad.parquet", index=False)
agresividad.round(4)

## 10. Qué pasa y qué no

Resumen de las pruebas de este notebook: v1 (12 oct), condiciones 1–3 con la escala de los controles y v2 completa (con FDR). La fila **placebo** es la tasa de falsos positivos: lo que pasa ahí es azar.

In [ ]:
def fila(analisis: str, t: pd.DataFrame) -> dict:
    v1 = t["pasa_v1"] if "pasa_v1" in t else t["pasa_1a3"]
    return {"análisis": analisis, "pruebas": len(t), "IC sin cero": int(t["ic_sin_cero"].sum()), "pasan v1": int(v1.sum()),
            "pasan 1–3 (escala v2)": int(t["pasa_1a3"].sum()), "pasan v2 (con FDR)": int(t["pasa"].sum()),
            "v2 con 10+ partidos": int((t["pasa"] & ~t["muestra_chica"].astype(bool)).sum()) if "muestra_chica" in t else int(t["pasa"].sum())}


resumen = pd.DataFrame([fila(f"Efecto de quiebres con {NUEVO}", jardine), fila(f"{NUEVO} frente a predecesores", contra_previos),
                        fila(f"Agresividad ({NUEVO})", agresividad[agresividad["grupo"] == NUEVO]),
                        fila(f"Placebo ({NUEVO}, quiebres falsos)", placebo)])
resumen["por azar en la condición 1 (≈5 %)"] = (resumen["pruebas"] * 0.05).round(1)

estructura = set(minuto.METRICAS["estructura"])
voronoi_pasa = pd.concat([jardine, contra_previos])
voronoi_pasa = voronoi_pasa[voronoi_pasa["metrica"].isin(estructura) & voronoi_pasa["pasa"]]
print("Voronoi y forma (bloque de estructura):",
      "SIGUE como lente estructural" if len(voronoi_pasa) else "queda como APOYO VISUAL (ninguna métrica pasa)")
resumen

In [ ]:
pasan = pd.concat([
    jardine[jardine["pasa"]].assign(análisis=f"Quiebre con {NUEVO}", valor=lambda d: d["efecto"]),
    contra_previos[contra_previos["pasa"]].assign(análisis=f"{NUEVO} − predecesores", valor=lambda d: d["diferencia"]),
], ignore_index=True)
pasan["métrica"] = pasan["metrica"].map(NOMBRES)
pasan[["análisis", "tipo", "lado", "métrica", "valor", "ic_bajo", "ic_alto", "tamano_de", "torneos_misma_direccion",
       "q", "muestra_chica"]].sort_values(["muestra_chica", "análisis", "tipo"]).round(3)

## Lectura

Corrida del 18 oct con el **criterio v2** (AP21–AP25, sin el CL26). Los números cambian si cambian los datos o `params.yaml`; la lectura de la v1 (12 oct) queda en el historial de git.

1. **El criterio v2 sí filtra.**
    - **Placebo:** en 280 pruebas sobre quiebres falsos, la v1 aprobaba 11 y la v2 aprueba 0. Esa es la tasa de falsos positivos del método.
    - **Pruebas reales:**

      | | v1 | v2 |
      | --- | --- | --- |
      | Quiebres con Jardine | 32 | 15 (13 con 10 o más partidos) |
      | Jardine frente a predecesores | 30 | 8 (6 con 10 o más partidos) |
2. **El método lee bien el partido.** Las expulsiones a favor mueven todo en la dirección esperada: más field tilt y control en campo rival, y el rival hunde y compacta su bloque. Cualquier equipo con uno más hace eso, así que no es un principio de Jardine. Es la prueba de que las ventanas y los controles funcionan.
3. **Lo que distingue a Jardine** (frente a Solari y Ortiz, mismo quiebre, misma lectura, criterio v2):
    - **Con uno más, hunde al rival.** Tras una expulsión a favor, el bloque del rival queda 19.5 yd más atrás que contra los predecesores (1.4 DE, 5 de 5 torneos). Además, el América concede 0.06 tiros en transición menos por minuto y contrapresiona menos: se instala arriba sin partirse.
    - **Una lesión no lo frena.** Tras una sustitución por lesión propia, presiona 1.0 vez más por minuto que los predecesores (1.1 DE, 5 de 5).
    - **Tras recibir gol, el rival no lo aprieta.** El rival contrapresiona 0.26 veces menos por minuto que contra los predecesores (0.6 DE, 5 de 5).
    - **Alerta:** sus cambios de sistema van seguidos de menos xG en los 10 minutos siguientes (−0.021 por minuto, 0.8 DE, 5 de 5). El cambio de sistema suele responder a algo que ya pasaba (selección), así que es una pista para el cuerpo técnico, no un efecto causal.
4. **Lo que ya no pasa.** Tres candidatos de la v1 no sobreviven a la escala de los controles o a la corrección por pruebas múltiples:
    - *ganando, cierra la transición*;
    - *reacciona al gol en contra* (OBV);
    - *sus cambios ganan espacio arriba* (Voronoi tras sustituciones tácticas).

    Se informan como "no pasa v2". La agresividad tampoco pasa: la presión alta no es identidad de Jardine.
5. **Voronoi** (auditoría A3):
    - El control en campo rival solo pasa tras expulsiones, y eso es mecánico: mide que hay un jugador menos.
    - Frente a los predecesores, lo que pasa es la **forma** del bloque del rival tras una expulsión.
    - Por la regla fijada, el bloque de estructura sigue como lente, pero el principio se apoya en la forma, y el Voronoi queda como la imagen y la medición del espacio.
    - Falta controlar el número de jugadores visibles (A4).
6. **Cautelas.**
    - Todo es comparación emparejada (diferencia en diferencias), no causalidad estricta: sustituir o cambiar de sistema es una decisión que responde a lo que pasaba.
    - Los candidatos se revisan al final en el torneo que el análisis no vio.